# Model Training Pipeline

This notebook handles:
- Loading preprocessed data
- Training multiple ML models
- Model evaluation and comparison
- Hyperparameter tuning
- Model saving

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score
import pickle
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

## 1. Load Preprocessed Data

In [ ]:
X_train = pd.read_csv('data/X_train.csv')
X_test = pd.read_csv('data/X_test.csv')
y_train = pd.read_csv('data/y_train.csv').squeeze()
y_test = pd.read_csv('data/y_test.csv').squeeze()

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
print(f'Target (train): {y_train.shape}')
print(f'Target (test): {y_test.shape}')

## 2. Train Multiple Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    results[name] = {
        'model': model,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'CV_R2_mean': cv_scores.mean(),
        'CV_R2_std': cv_scores.std()
    }
    
    print(f'  R² Score: {r2:.4f}')
    print(f'  RMSE: {rmse:.4f}')
    print(f'  MAE: {mae:.4f}')
    print(f'  CV R² (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}\n')

## 3. Model Comparison

In [ ]:
comparison_df = pd.DataFrame({
    'Model': results.keys(),
    'R² Score': [results[m]['R2'] for m in results.keys()],
    'RMSE': [results[m]['RMSE'] for m in results.keys()],
    'MAE': [results[m]['MAE'] for m in results.keys()],
    'CV R² (mean)': [results[m]['CV_R2_mean'] for m in results.keys()]
})

comparison_df = comparison_df.sort_values('R² Score', ascending=False)
print('\n=== MODEL COMPARISON ===')
print(comparison_df.to_string(index=False))

## 4. Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ['R² Score', 'RMSE', 'MAE']
for idx, metric in enumerate(metrics):
    sorted_df = comparison_df.sort_values(metric, ascending=(metric != 'R² Score'))
    axes[idx].barh(sorted_df['Model'], sorted_df[metric], color='#ed53a3')
    axes[idx].set_xlabel(metric, fontweight='bold')
    axes[idx].set_title(f'{metric} by Model', fontweight='bold')

plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved model comparison plot')

## 5. Feature Importance Analysis

In [ ]:
best_model_name = comparison_df.iloc[0]['Model']
best_model = results[best_model_name]['model']

print(f'\nBest Model: {best_model_name}')
print(f'R² Score: {results[best_model_name]["R2"]:.4f}')

if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['feature'][:15], feature_importance['importance'][:15], color='#ed53a3')
    plt.xlabel('Importance', fontweight='bold')
    plt.title(f'Top 15 Features - {best_model_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig('results/feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('\nTop 10 Features:')
    print(feature_importance.head(10).to_string(index=False))

## 6. Save Best Model

In [ ]:
with open('models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('models/best_model_name.txt', 'w') as f:
    f.write(best_model_name)

print(f'✓ Saved best model: {best_model_name}')

## 7. Training Summary

In [ ]:
print('\n=== TRAINING SUMMARY ===')
print(f'\nBest Performing Model: {best_model_name}')
print(f'  R² Score: {results[best_model_name]["R2"]:.4f}')
print(f'  RMSE: {results[best_model_name]["RMSE"]:.4f}')
print(f'  MAE: {results[best_model_name]["MAE"]:.4f}')
print(f'  CV R² (mean): {results[best_model_name]["CV_R2_mean"]:.4f}')
print(f'\nTraining Set Size: {len(X_train)}')
print(f'Test Set Size: {len(X_test)}')
print(f'Number of Features: {X_train.shape[1]}')
print(f'\nModels trained: {len(models)}')